In [2]:
!pip install mediapipe opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [3]:
!pip uninstall mediapipe protobuf -y

Found existing installation: mediapipe 0.10.35
Uninstalling mediapipe-0.10.35:
  Successfully uninstalled mediapipe-0.10.35
Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6


In [4]:
!pip install mediapipe protobuf tensorflow -y


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -y


In [5]:
!pip uninstall tensorflow -y
!pip uninstall protobuf mediapipe -y

!pip install mediapipe==0.10.21
!pip install protobuf==4.25.3


Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf-tf 2.20.0 requires tensorflow==2.20.0, which is not installed.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.3 which is incompatible.
grpc-google-iam-v1 0.14.4 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-language 2.20.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-secret-manager 2.27.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-logging 3.15.0 requires protobuf<8.0.0,>=4.25.

In [8]:
import cv2
import mediapipe as mp
import numpy as np
from google.colab.patches import cv2_imshow
from google.colab import files
import requests
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Calculate the angle between three given points
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))
    return angle

In [10]:
# Initialize MediaPipe Pose estimation
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

input_video = '/content/Dale steyn bowling action slow motion.mp4'
output_video = '/content/drive/MyDrive/ai_coach_final_dashboard.avi'
cap = cv2.VideoCapture(input_video)

In [11]:
if not cap.isOpened():
    print("Error: Could not open video file. Please check the file path.")
else:
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # Define codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    print("Processing video frames...")

    # State variables for bowling action tracking
    frame_count = 0
    bowler_state = "RUNNING"
    landing_knee_angle = 0
    min_wrist_y = 9999
    release_elbow_angle = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # Convert to RGB and copy to ensure a contiguous array for MediaPipe
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).copy()

        # Process the frame to find pose landmarks
        results = pose.process(frame_rgb)

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # 1. Head Drop Calculation
            nose = landmarks[mp_pose.PoseLandmark.NOSE]
            left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER]
            right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER]
            nose_y = int(nose.y * height)
            shoulder_y = int((left_shoulder.y + right_shoulder.y) / 2 * height)
            head_distance = shoulder_y - nose_y

            # 2. Ankle Tracking for State Machine
            left_ankle = landmarks[mp_pose.PoseLandmark.LEFT_ANKLE]
            ankle_y = int(left_ankle.y * height)

            # 3. Bowling State Machine (Running -> Jumping -> Landed)
            if ankle_y < 280:
                bowler_state = "JUMPING"

            elif bowler_state == "JUMPING" and ankle_y > 330:
                bowler_state = "LANDED"

                # Calculate knee angle exactly at the landing frame
                hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP].y]
                knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE].y]
                ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE].y]
                landing_knee_angle = calculate_angle(hip, knee, ankle)
                print(f"Landing stride detected. Knee Angle: {int(landing_knee_angle)} degrees")

            # 4. Ball Release & Elbow Extension Logic
            if bowler_state == "LANDED":
                right_wrist = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST]
                wrist_y = int(right_wrist.y * height)

                # Track the highest point of the wrist to determine release point
                if wrist_y < min_wrist_y:
                    min_wrist_y = wrist_y
                    r_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER].y]
                    r_elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW].y]
                    r_wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y]
                    release_elbow_angle = calculate_angle(r_shoulder, r_elbow, r_wrist)

            # Draw pose landmarks on the frame
            mp_draw.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                mp_draw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2),
                mp_draw.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2))

            # UI Dashboard setup (Semi-transparent background)
            overlay = frame.copy()
            cv2.rectangle(overlay, (20, 20), (550, 260), (0, 0, 0), -1)
            alpha = 0.4
            frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

            # Display metrics on the dashboard
            if head_distance < 80:
                cv2.putText(frame, f'Head: DROP ({head_distance})', (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            else:
                cv2.putText(frame, f'Head: STABLE ({head_distance})', (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            cv2.putText(frame, f'Ankle Y: {ankle_y}', (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            cv2.putText(frame, f'STATE: {bowler_state}', (30, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 100, 100), 2)

            if landing_knee_angle > 0:
                cv2.putText(frame, f'Knee Angle: {int(landing_knee_angle)} deg', (30, 180), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

            if release_elbow_angle > 0:
                cv2.putText(frame, f'Elbow Release: {int(release_elbow_angle)} deg', (30, 220), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 100, 100), 2)

        out.write(frame)
        frame_count += 1

        # Print progress every 30 frames
        if frame_count % 30 == 0:
            print(f"Processed {frame_count} frames...")

    cap.release()
    out.release()
    print(f"\nVideo processing complete. Output saved to: '{output_video}'")

Processing video frames...
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Landing stride detected. Knee Angle: 169 degrees
Processed 300 frames...
Processed 330 frames...

Video processing complete. Output saved to: '/content/drive/MyDrive/ai_coach_final_dashboard.avi'


In [7]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import files
files.download('/content/drive/MyDrive/ai_coach_final_dashboard.avi')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>